# QC200 controlled training

Public workflow notes for the frozen 200 Hz, 12-second DGDCN adaptation on TUSZ v2.0.6. The paper used v2.0.1, so this is not a claim of exact reproduction. This notebook has no run outputs, credentials, Drive identifiers, markers, EEG features, checkpoints, or predictions.

B1 and W1 are separate fresh runs. Historical H0 already used patient-balanced sampling. The current controlled pair focuses on reviewed QC markers, fresh verified normalization, strict validation, and a controlled loss-weight change. No new controlled-run metrics are reported.


## Source and upstream model restore

Start in a fresh Colab runtime. The first code cell downloads this public backup. The next cell restores only the required upstream reference files from pinned commit 96c7bee49c240124735ffc2ab2ad118d27a1ff97 and applies documented compatibility repairs. The upstream snapshot has no identified license; the runtime restore does not grant redistribution rights.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/S-Srinivasan-06/Research-progress-documentation.git"
REPO_ROOT = Path("/content/Research-progress-documentation")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

CODE = REPO_ROOT / "experiments" / "qc200-training"
RUNTIME = REPO_ROOT / "experiments" / "qc200-training-runtime"
assert CODE.is_dir() and RUNTIME.is_dir(), "The public backup checkout is incomplete."


In [ ]:
subprocess.run(
    [sys.executable, str(RUNTIME / "restore_reference.py"),
     "--destination", str(CODE / "repo")],
    check=True,
)


## Configure private inputs

Mount Drive and authenticate the Google Drive API in this runtime before launching the staged B1 pipeline. Create a private config from the example and edit it in the runtime only. Supply verified feature-cache and QC marker paths, run/output locations, the existing Drive parent, and the private archive/reference manifest paths required by the staged B1 API. Do not commit the edited config.

Keep feature provenance unapproved until the representative source/cache comparison and full cache audit have passed. The staged pipeline requires a fresh B1 run folder. A positive preprocessing limit is pilot-only and cannot approve a training cache.


In [ ]:
from google.colab import auth
import json
import shutil

auth.authenticate_user()

B1_CONFIG = Path("/content/qc200-B1.private.json")
if not B1_CONFIG.exists():
    shutil.copy2(CODE / "config.B1.example.json", B1_CONFIG)
print("Private B1 config template:", B1_CONFIG)


## B1: staged restore, audit, and train/dev

The staged API restores the existing archive, compares representative source features, runs full feature validation, then starts B1 with Drive persistence and read-back checks. It does not evaluate the held-out partition. The archive restore reached 40/40 files, and the representative source/cache comparison passed. The Colab runtime was later reset, so full feature audit completion is unconfirmed. No B1 training metrics or checkpoint are available. The user has reconnected on CPU and reauthorized Drive, but recovery is not confirmed. The W1 queue helper is prepared but has not been deployed.


In [ ]:
START_B1 = False  # Set True only after editing and reviewing the private config.
if START_B1:
    subprocess.run(
        [sys.executable, str(CODE / "execute_pipeline.py"),
         "--config", str(B1_CONFIG)],
        check=True,
    )


## W1: a separate run

W1 needs its own run ID and new output folder. Keep the seed, data, markers, architecture, augmentation, normalization population, and schedule fixed; change only variant. B1 uses unweighted BCE. W1 computes positive-class loss weight from the exact selected training pool. The current automated staging pipeline covers B1; automatic B1-to-W1 queue sequencing is still in progress.


In [ ]:
W1_CONFIG = Path("/content/qc200-W1.private.json")
if not W1_CONFIG.exists():
    w1 = json.loads(B1_CONFIG.read_text())
    w1["run_id"] = "SET-A-UNIQUE-W1-RUN-ID"
    w1["variant"] = "W1"
    w1["output_dir"] = "/content/drive/MyDrive/SET-A-NEW-W1-RUN-FOLDER"
    W1_CONFIG.write_text(json.dumps(w1, indent=2) + "\n")

START_W1 = False  # Review every private path and the unique run identity first.
if START_W1:
    subprocess.run(
        [sys.executable, str(CODE / "train_qc200.py"),
         "--config", str(W1_CONFIG)],
        check=True,
    )


## Held-out evaluation and recovery limits

Held-out evaluation is a separate explicit call to finalize_qc200.py. It requires completed train/dev status, a frozen development-only selection record, a matching run/checkpoint, and disclosure of earlier evaluation access. Training does not run eval.

The optional cpu_fallback_watch.py helper can resume an eligible GPU device/driver failure from last.pt on CPU while the original Colab runtime and watcher remain alive. It requires explicit matching failure evidence, an existing verified Drive run folder, no live trainer, and enough time before the supplied runtime deadline. It does not recover arbitrary code/data errors or reconnect after runtime termination. Use the helper README for its exact CLI arguments.

Clip classification does not provide event sensitivity, false alarms per hour, or detection latency. Those need a validated continuous event-scoring protocol. Do not interpret this notebook as a clinical claim.


## Queue W1 after B1
Run this supervisor in the background while B1 is active. Set an absolute deadline for this runtime. The queue needs validated private inputs, working Drive authorization and enough time after B1 finishes. A full runtime reset stops both jobs.


In [ ]:
QUEUE_W1 = False
W1_RUN_ID = "SET-A-UNIQUE-W1-RUN-ID"
DEADLINE_UTC = "SET-AN-ISO-UTC-DEADLINE"
if QUEUE_W1:
    queue_log = open("/content/w1-queue.log", "a", buffering=1)
    queue = subprocess.Popen(
        [sys.executable, str(RUNTIME / "queue_w1.py"),
         "--b1-config", str(B1_CONFIG), "--w1-run-id", W1_RUN_ID,
         "--deadline-utc", DEADLINE_UTC, "--source-dir", str(CODE)],
        stdout=queue_log, stderr=subprocess.STDOUT, start_new_session=True)
    print("W1 queue PID:", queue.pid)
